# 02 -- Production envelopes and growth coupling

Notebook 01 asked what the model predicts at **one** growth rate: the optimum. Here
we look at the whole range:

1. a **production envelope**: how much product is possible at each growth rate,
2. **growth coupling**: knockouts that force the cell to make a product in order to
   grow.

Same notation as before: fluxes in mmol gDW^-1 h^-1, growth in h^-1, uptake
negative.

## Part 0 -- Setup

In [ ]:
# Setup -- run this cell first. Safe to re-run: it only installs what is missing.
# If Colab asks you to restart the session after installing, do so and run it again.
import importlib.util
import sys

IN_COLAB = "google.colab" in sys.modules

if importlib.util.find_spec("cobra") is None:
    %pip install -q cobra==0.32.1
if importlib.util.find_spec("escher") is None:
    %pip install -q escher==1.8.1  # optional flux maps; cobra works without it

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()  # lets Colab draw third-party widgets

import cobra
from cobra.io import load_model

_mu = load_model("textbook").slim_optimize()
assert 0.8 < _mu < 0.95, f"Smoke test failed: growth rate {_mu}"
print(f"cobra {cobra.__version__} | Colab: {IN_COLAB} | smoke test {_mu:.4f} h^-1")
print("Setup OK")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from cobra.flux_analysis import flux_variability_analysis, production_envelope

model = load_model("textbook")
BIOMASS = "Biomass_Ecoli_core"

## Part 1 -- Production envelope

A production envelope puts **growth on the x-axis and a product on the y-axis**.
For each growth rate it shows the minimum and maximum product flux the network
allows. The culture is **anaerobic** (oxygen bound 0), where
*E. coli* has to get rid of NADH by making fermentation products. The product is
succinate.

In [ ]:
model.reactions.EX_o2_e.lower_bound = 0  # anaerobic
TARGET = "EX_succ_e"

wt_env = production_envelope(model, [BIOMASS], objective=TARGET, points=25)
wt_env[[BIOMASS, "flux_minimum", "flux_maximum"]].round(3).head()

In [ ]:
def plot_envelope(env, ax, label, color):
    ax.fill_between(
        env[BIOMASS],
        env["flux_minimum"],
        env["flux_maximum"],
        color=color,
        alpha=0.3,
        label=label,
    )
    ax.plot(env[BIOMASS], env["flux_minimum"], color=color)
    ax.plot(env[BIOMASS], env["flux_maximum"], color=color)


fig, ax = plt.subplots(figsize=(6, 4))
plot_envelope(wt_env, ax, "wild type", "tab:blue")
ax.set_xlabel("growth rate (h$^{-1}$)")
ax.set_ylabel("succinate secretion (mmol gDW$^{-1}$ h$^{-1}$)")
ax.legend()
plt.show()

Read it corner by corner:

- the **top-left corner** is the theoretical maximum: all carbon to succinate, no
  growth;
- the **right-hand end** is maximum growth, which is what plain FBA predicts;
- the **lower edge sits on zero** all the way along: the wild type *can* make
  succinate but never *has to*. At maximum growth nothing forces it to, so FBA
  predicts none.

In [ ]:
v_glc = -model.reactions.EX_glc__D_e.lower_bound
top = wt_env.loc[wt_env[BIOMASS].idxmin()]
end = wt_env.loc[wt_env[BIOMASS].idxmax()]
print(f"glucose uptake: {v_glc:.1f} mmol gDW^-1 h^-1")
print(
    f"max succinate at zero growth: {top.flux_maximum:.2f} -> "
    f"Y_p/s = {top.flux_maximum / v_glc:.2f} mol/mol"
)
print(
    f"at max growth ({end[BIOMASS]:.3f} h^-1): succinate between "
    f"{end.flux_minimum:.2f} and {end.flux_maximum:.2f}"
)

## Part 2 -- Growth coupling with knockouts

Where does the carbon go instead? FVA at maximum growth on all the fermentation
products tells us which outlets the wild type uses.

In [ ]:
BYPRODUCTS = ["EX_ac_e", "EX_etoh_e", "EX_for_e", "EX_lac__D_e", "EX_succ_e"]


def at_max_growth(m):
    """Growth rate and by-product ranges when growth is at its maximum."""
    mu = m.slim_optimize(error_value=0.0)
    fva = flux_variability_analysis(m, BYPRODUCTS, fraction_of_optimum=1.0)
    return mu, fva.round(3) + 0.0


mu_wt, fva_wt = at_max_growth(model)
print(f"wild type, anaerobic: growth {mu_wt:.4f} h^-1")
fva_wt

Acetate, ethanol and formate: classic mixed-acid fermentation. Ethanol is the main
way to reoxidize NADH. Block it by knocking out acetaldehyde dehydrogenase
(`ACALD`), the step that feeds ethanol formation. `with model:` keeps the knockout
temporary.

In [ ]:
for rxn_id in ["ACALD", "LDH_D"]:
    rxn = model.reactions.get_by_id(rxn_id)
    genes = ", ".join(g.name for g in rxn.genes)
    print(f"{rxn.id:6s} {rxn.name}  |  GPR: {rxn.gene_reaction_rule}  ({genes})")

with model:
    model.reactions.ACALD.knock_out()
    mu_1, fva_1 = at_max_growth(model)
print(f"\nΔACALD: growth {mu_1:.4f} h^-1")
fva_1

The cell found another outlet for NADH: lactate. Close that one too with
D-lactate dehydrogenase (`LDH_D`).

In [ ]:
KNOCKOUTS = ["ACALD", "LDH_D"]

with model:
    for rxn_id in KNOCKOUTS:
        model.reactions.get_by_id(rxn_id).knock_out()
    mu_2, fva_2 = at_max_growth(model)
    ko_env = production_envelope(model, [BIOMASS], objective=TARGET, points=25)
print(f"ΔACALD ΔLDH_D: growth {mu_2:.4f} h^-1")
fva_2

Now succinate has a minimum above zero at maximum growth: the only way left for the
cell to balance its redox *and* grow is to make succinate. That is **growth
coupling**. Compare the envelopes:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_envelope(wt_env, ax, "wild type", "tab:blue")
plot_envelope(ko_env, ax, "ΔACALD ΔLDH_D", "tab:red")
ax.set_xlabel("growth rate (h$^{-1}$)")
ax.set_ylabel("succinate secretion (mmol gDW$^{-1}$ h$^{-1}$)")
ax.legend()
plt.show()

lowest = ko_env["flux_minimum"].min()
end_ko = ko_env.loc[ko_env[BIOMASS].idxmax()]
print(f"mutant: succinate is at least {lowest:.2f} at every growth rate")
print(
    f"at max growth ({end_ko[BIOMASS]:.3f} h^-1): succinate {end_ko.flux_minimum:.2f}"
    f" -> Y_p/s = {end_ko.flux_minimum / v_glc:.2f} mol/mol"
)
print(
    f"growth cost of the design: {mu_wt:.3f} -> {mu_2:.3f} h^-1 "
    f"({100 * (1 - mu_2 / mu_wt):.0f}% lower)"
)

Three things to take from this plot:

- the **lower edge lifted off zero**: the mutant must secrete succinate at every
  growth rate, not only at the maximum. This is *strong* coupling. Coupling only at
  maximum growth would be *weak* coupling.
- the **top edge barely moved**: the knockouts did not create new capacity, they
  removed alternatives.
- **coupling is not free**: maximum growth dropped. Under adaptive laboratory
  evolution the mutant can only grow faster by moving right along its envelope, and
  that means making more succinate.

**A caveat about genes.** We knocked out *reactions*. The GPRs above say that each
of these reactions has two genes joined by `or` (isozymes), so in the lab each
reaction needs both genes deleted -- unless one isozyme is not really active under
these conditions, which the model cannot tell you. This is where a model
prediction meets real biology.

## Your turn -- can you find a design yourself?

Real design tools (OptKnock, minimal cut sets in StrainDesign) search knockouts
with clever optimization. On a model this small we can simply try every pair. The
cell below tests all double knockouts among reactions that have genes and are not
essential, and keeps the ones that couple `TARGET` to growth.

Things to try:

1. Run it for succinate and check that `ACALD` + `LDH_D` is among the hits. What
   else turns up, and do the hits make biological sense?
2. Change `TARGET` to lactate (`EX_lac__D_e`), ethanol (`EX_etoh_e`) or acetate
   (`EX_ac_e`). Which products are easy to couple, which are hard?
3. Switch oxygen back on (`O2_BOUND = -1000`). Does coupling get easier or harder,
   and why?
4. Repeat the gene caveat in the model: instead of reactions, knock out genes with
   `model.genes.get_by_id("b1241").knock_out()` (adhE) and `"b2133"` (ldhA). Is the
   design still coupled? Why not?
5. Pick one hit, draw its production envelope like the one above, and decide: weak
   or strong coupling?

In [ ]:
import itertools

TARGET = "EX_succ_e"
O2_BOUND = 0  # 0 = anaerobic, -1000 = fully aerobic


def coupling(m, target):
    """Growth rate and the minimum product flux at maximum growth."""
    mu = m.slim_optimize(error_value=0.0)
    if mu < 0.01:
        return mu, 0.0
    with m:
        m.reactions.get_by_id(BIOMASS).lower_bound = mu * 0.999
        m.objective = target
        m.objective_direction = "min"
        return mu, m.slim_optimize(error_value=0.0)


hits = []
with model:
    model.reactions.EX_o2_e.lower_bound = O2_BOUND
    mu_ref = model.slim_optimize()
    candidates = []
    for rxn in model.reactions:
        if rxn.boundary or rxn.id in (BIOMASS, "ATPM") or not rxn.genes:
            continue
        with model:
            rxn.knock_out()
            if model.slim_optimize(error_value=0.0) > 0.1 * mu_ref:
                candidates.append(rxn.id)
    for pair in itertools.combinations(candidates, 2):
        with model:
            for rxn_id in pair:
                model.reactions.get_by_id(rxn_id).knock_out()
            mu, p_min = coupling(model, TARGET)
            if p_min > 1e-3:
                hits.append((" + ".join(pair), mu, p_min))

hits = pd.DataFrame(
    hits, columns=["knockouts", "growth", f"min {TARGET} at max growth"]
)
print(f"{len(candidates)} candidate reactions, {len(hits)} coupled pairs")
hits.sort_values(f"min {TARGET} at max growth", ascending=False).head(15).round(3)

In [ ]:
# your code here